# 03 – Batch inference

Loads the joblib model and scores:
- 2025 rows that have an actual next quarter (error can be computed)
- the latest quarter of each province and crop, which forecasts the quarter after the data ends

Predicted risk uses the forecast, not the historical `risk_label`:
- year-over-year growth <= -0.2 -> Declining
- climate stress >= 30 -> Risk-prone
- year-over-year growth >= 0.2 -> High
- otherwise -> Stable

Climate stress is abs(rolling rain anomaly) + abs(rolling temperature anomaly).
Year-over-year growth compares the forecast with the same quarter one year earlier
(three quarters before the feature row).

In [0]:
import joblib
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from configs.project_config import (
    GOLD_TABLES,
    ML_TABLES,
    MODEL_PATH,
    MODEL_NAME,
    TEST_YEAR,
    NUMERIC_FEATURES,
    BOOLEAN_FEATURES,
    CATEGORICAL_FEATURES,
)

feature_cols = NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES
DECLINE_THRESHOLD = -0.2
GROWTH_THRESHOLD = 0.2
STRESS_THRESHOLD = 30.0

In [0]:
saved = joblib.load(MODEL_PATH)
model = saved["model"]
print(f"Loaded {saved['model_name']} from {MODEL_PATH}")

In [0]:
w = Window.partitionBy("province", "crop").orderBy("time_index")
latest_w = Window.partitionBy("province", "crop")

features = (
    spark.table(GOLD_TABLES["ml_crop_forecast_features"])
    .withColumn("production_lag_3", F.lag("production", 3).over(w))
    .withColumn("max_time_index", F.max("time_index").over(latest_w))
)
labeled_test = features.where(f"year = {TEST_YEAR} AND production_next_quarter IS NOT NULL")
forward = features.where("time_index = max_time_index AND production_next_quarter IS NULL")
to_score = labeled_test.unionByName(forward)
pdf = to_score.toPandas()
for col in BOOLEAN_FEATURES:
    pdf[col] = pdf[col].astype("float")
if pdf.empty:
    raise ValueError("Nothing to score.")

pdf["predicted_production"] = np.asarray(model.predict(pdf[feature_cols]), dtype=float)
pdf["actual_production"] = pdf["production_next_quarter"]
pdf["error"] = pdf["predicted_production"] - pdf["actual_production"]
pdf["climate_stress"] = pdf["rain_anomaly_rolling"].abs() + pdf["temp_anomaly_rolling"].abs()

base = pdf["production_lag_3"]
pdf["predicted_yoy"] = np.where(
    base.notna() & (base != 0),
    (pdf["predicted_production"] - base) / base,
    np.nan,
)

def is_missing(value):
    return value is None or (isinstance(value, float) and np.isnan(value))

def risk_from_forecast(yoy, stress):
    if is_missing(yoy):
        return None
    if yoy <= DECLINE_THRESHOLD:
        return "Declining"
    if not is_missing(stress) and stress >= STRESS_THRESHOLD:
        return "Risk-prone"
    if yoy >= GROWTH_THRESHOLD:
        return "High"
    return "Stable"

pdf["predicted_risk"] = [
    risk_from_forecast(yoy, stress)
    for yoy, stress in zip(pdf["predicted_yoy"], pdf["climate_stress"])
]
pdf["forecast_year"] = np.where(pdf["quarter_num"] == 4, pdf["year"] + 1, pdf["year"])
pdf["forecast_quarter"] = np.where(
    pdf["quarter_num"] == 4,
    "Q1",
    "Q" + (pdf["quarter_num"] + 1).astype(int).astype(str),
)

output = pdf[[
    "province", "crop", "crop_group", "year", "quarter",
    "forecast_year", "forecast_quarter",
    "actual_production", "predicted_production", "error",
    "predicted_yoy", "climate_stress", "predicted_risk",
]].rename(columns={"year": "feature_year", "quarter": "feature_quarter"})
output["model_name"] = MODEL_NAME

result = spark.createDataFrame(output)
result = result.withColumn("scored_at", F.current_timestamp())

target = ML_TABLES["crop_production_predictions"]
(
    result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)
print(f"Wrote {target}")
print(f"Rows: {result.count()}")
result.groupBy("predicted_risk").count().orderBy("predicted_risk").show()